# Battery Health Prognostics - Quick Start

End-to-end demo: Load → Validate → Features → Train → Evaluate → Safety

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Load & Validate Data

In [ ]:
from src.data.unified_loader import UnifiedDataLoader
from src.data.validator import DataValidator

loader = UnifiedDataLoader()
df = loader.load_all(nasa_dir='../data/battery_data')
validator = DataValidator()
df, report = validator.validate(df)
print(f'Loaded {len(df)} cycles, {df["battery_id"].nunique()} batteries')
print(f'Validation pass rate: {report.pass_rate:.1%}')

## 2. Feature Extraction

In [ ]:
from src.features.extractor import FeatureExtractor

extractor = FeatureExtractor()
df = extractor.extract_all(df)
feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in ('cycle', 'rul')]
df = df.dropna(subset=feature_cols + ['rul']).reset_index(drop=True)
print(f'{len(feature_cols)} features, {len(df)} rows')

## 3. Degradation Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for bat in df['battery_id'].unique():
    sub = df[df['battery_id'] == bat].sort_values('cycle')
    ax.plot(sub['cycle'], sub['capacity'], label=bat, linewidth=2)
ax.set_xlabel('Cycle'); ax.set_ylabel('Capacity (Ah)')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()

## 4. Train & Predict

In [ ]:
from src.models import LSTMModel
from src.data.splitter import DataSplitter
from src.uncertainty.scoring import compute_all_metrics

model = LSTMModel(input_dim=len(feature_cols), hidden_dim=64, seq_length=30, epochs=50)

for train_df, test_df, test_id in DataSplitter.logo_cv(df):
    model.fit(train_df[feature_cols].values, train_df['rul'].values)
    mean, lower, upper = model.predict(test_df[feature_cols].values)
    y_eval = test_df['rul'].values[-len(mean):]
    metrics = compute_all_metrics(y_eval, mean, lower, upper)
    print(f'{test_id}: RMSE={metrics["RMSE"]:.2f}, CRPS={metrics["CRPS"]:.2f}, PICP={metrics["PICP"]:.2f}')

## 5. Safety Decision

In [ ]:
from src.safety.decision_engine import SafetyDecisionEngine

engine = SafetyDecisionEngine()
decisions = engine.decide_batch(mean, lower, upper)
for d in decisions[-5:]:
    print(f'RUL={d.rul_estimate:.0f} [{d.confidence_lower:.0f},{d.confidence_upper:.0f}] → {d.level.value}: {d.action}')